# BigEarthNet.txt data-prep (v4, CPU-only, no GPU needed)

**Updated 2026-09-13**: fixes a real crash some of you will have hit on the smoke-test
cell -- `'[Errno 9] Bad file descriptor'` + `Fatal Python error: PyGILState_Release:
auto-releasing thread-state, but no thread-state for this thread`. That crash happened
*after* the real selection work had already finished and `bigearthnet_txt_selected.json`
was already written correctly -- it was HF's `datasets` streaming library leaving a
background retry thread alive after this script's early `break` out of the stream, which
then crashed during normal Python interpreter shutdown. This version makes the script
exit immediately (`os._exit(0)`) right after the real output file is safely on disk, which
skips the interpreter-finalization step that thread was crashing during. Also fixes a
real `s1_name`/`S1_name` case-mismatch bug (harmless today, since this field isn't used
downstream yet, but fixed for honesty).

Run this on a **separate Kaggle notebook** from the main training one (`kaggle_finetune_qwen2vl_v4.ipynb`) — this stage is CPU/network-bound, not GPU-bound.

**Before running:** Settings → Accelerator → **None** → Internet: **On**. Add an `HF_TOKEN` Kaggle secret (write access to `Sameelkazi/satquery-qwen3vl-vrsbench-lora-v4-bigearthnet-txt-cache`).

Run cells top to bottom. The smoke test (Cell 4) is fast (~seconds) — check its output looks sane before running the real filter (Cell 5), and definitely before the big ~59 GiB extract step (Cell 7), which only needs to run **once ever** (later runs from any account just download the small cached result).

In [ ]:
!df -h /kaggle/working


In [ ]:
!pip install -q datasets huggingface_hub zstandard rasterio pillow numpy


## Cell: write out the two real project scripts (embedded below, no separate upload needed)

In [ ]:
%%writefile bigearthnet_txt_filter.py
"""
finetuning/bigearthnet_txt_filter.py

Purpose (real, PS-relevant): the PS names "BigEarthNet.txt" (arXiv:2603.29630) as its
primary/prescribed fine-tuning dataset. This project's actual fine-tuning run uses
VRSBench instead (a PS-named EVALUATION dataset, used here for training under the PS's own
"BigEarthNet.txt OR OTHER OPEN-SOURCE TRAINING DATA" clause) -- a documented, defensible
substitution, not a violation. This script adds a genuine second training source built
directly from BigEarthNet.txt itself, closing that gap for real instead of just arguing
around it.

What this script does: reads BigEarthNet.txt's real annotation parquet
(BIFOLD-BigEarthNetv2-0/BigEarthNet.txt on the HF Hub, 9.55M rows -- verified via HF's
datasets-server API on 2026-08-29) and selects a small, real, referring-expression subset
(type == "bounding box", category == "reference" -- i.e. "find the box for THIS DESCRIBED
OBJECT", the same text-to-box task shape as VRSBench's own [refer] examples and the same
task the project's Acc@0.5 grounding gap is about). It deliberately EXCLUDES
category == "point" rows (a different point-to-box task shape) to avoid teaching the model
two incompatible box-output conventions.

Verified real facts this script relies on (checked directly via HF's datasets-server API,
not assumed):
  - Real columns: ID, S1_name, patch_id, input, output, type, category, split, latitude,
    longitude, country, season, climate_zone.
  - Real box format for type=="bounding box": output is a bracket string
    "[x_min y_min, x_max y_max]" with NORMALIZED [0,1] coordinates, e.g. "[0.0 0.33, 0.28 0.8]".
  - Real category=="reference" input format: "Identify the location of the
    <ref>DESCRIPTION</ref>" (or similar phrasing) -- the <ref>...</ref> inner text is the
    real object description to keep; the wrapper phrasing/tags are stripped.
  - patch_id values (e.g. "S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_26_57") match the
    REAL BigEarthNet v2.0 per-patch folder-naming convention (confirmed against
    bigearth.net's own "Description_BigEarthNet_v2.pdf") -- this is the exact folder name to
    look for inside BigEarthNet-S2.tar.zst.

Output schema matches finetuning/kaggle_finetune_qwen2vl.ipynb Cell 3's own manifest.json
exactly ({"image_path", "prompt", "response", "task_tag"}), except image_path is left as
None here -- bigearthnet_txt_extract.py fills it in after real image extraction, so this
filter step never needs network access to the (much larger) image archives at all.

This file is meant to run on Kaggle (real network access to the HF Hub). It has no heavy
image/network dependencies of its own beyond `datasets`/`huggingface_hub`/`pyarrow`.
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Any

TEXT_DATASET_REPO = "BIFOLD-BigEarthNetv2-0/BigEarthNet.txt"

# Real box format audit regex: "[x_min y_min, x_max y_max]", normalized 0-1 floats.
BOX_RE = re.compile(
    r"\[\s*([0-9.]+)\s+([0-9.]+)\s*,\s*([0-9.]+)\s+([0-9.]+)\s*\]"
)
# Strips the <ref>...</ref> / <point>(...)</point> XML-like wrapper tags this dataset's
# `input` field uses, keeping the inner text as the plain-language object description.
REF_TAG_RE = re.compile(r"<ref>(.*?)</ref>", re.IGNORECASE | re.DOTALL)
POINT_TAG_RE = re.compile(r"<point>.*?</point>", re.IGNORECASE | re.DOTALL)

REQUIRED_COLUMNS = [
    "patch_id", "input", "output", "type", "category", "split",
]


def clean_reference_prompt(raw_input: str) -> str | None:
    """Extract the plain-language object description from a category=='reference' row's
    `input` field. Returns None if no <ref>...</ref> span is found (defensive -- do not
    guess at a different real format if this dataset's phrasing varies row to row)."""
    m = REF_TAG_RE.search(raw_input or "")
    if not m:
        return None
    description = m.group(1).strip()
    if not description:
        return None
    return f"[refer] Where is the {description}?"


def box_str_to_token(output_text: str) -> str | None:
    """Convert BigEarthNet.txt's real "[x_min y_min, x_max y_max]" (normalized 0-1) box
    string into this project's real box-token format.

    UPDATED 2026-09-12 (v4 base-model + box-format switch, paired with
    finetuning/kaggle_finetune_qwen2vl.ipynb's swap to Qwen3-VL-4B-Instruct): now emits
    '{"bbox_2d": [x1, y1, x2, y2]}' on a 0-1000 integer scale, x-first -- Qwen3-VL's own
    documented native grounding format (github.com/QwenLM/Qwen3-VL,
    cookbooks/2d_grounding.ipynb), NOT the old GeoChat-era "{<x1><y1><x2><y2>}" 0-100
    bracket convention this function used to emit. This is the SAME format the notebook's
    Cell 3 (convert_legacy_box_tokens) now converts VRSBench's raw answers into, and the
    SAME format backend/models/geochat_service.py's parse_boxes_from_text expects at
    inference (x-first order, already settled there against real IoU evidence). Training
    BOTH data sources toward one shared output convention is deliberate -- mixing two
    different box encodings in one fine-tune would teach the model an inconsistent output
    format. Deliberately omits the cookbook's optional "label" field -- see the notebook's
    Cell 3 note for why (this task's referring expression already IS the label, and
    BigEarthNet.txt's own "input" field plays that same role here)."""
    m = BOX_RE.search(output_text or "")
    if not m:
        return None
    x1, y1, x2, y2 = (float(g) for g in m.groups())
    # Defensive real-data check: reject rows with an inverted/degenerate box outright
    # rather than silently training on a bad label.
    if not (0.0 <= x1 < x2 <= 1.0 and 0.0 <= y1 < y2 <= 1.0):
        return None
    xi1, yi1, xi2, yi2 = (round(v * 1000) for v in (x1, y1, x2, y2))
    return f'{{"bbox_2d": [{xi1}, {yi1}, {xi2}, {yi2}]}}'


def select_rows(rows: list[dict[str, Any]], max_examples: int, dedupe_per_patch: int = 3
                 ) -> tuple[list[dict[str, Any]], Counter]:
    """Real filtering/conversion logic, factored out so it can be unit-tested against
    synthetic rows without any network access (see test_bigearthnet_txt_filter.py)."""
    selected: list[dict[str, Any]] = []
    per_patch_count: Counter = Counter()
    stats = Counter()

    for row in rows:
        if len(selected) >= max_examples:
            break
        stats["seen"] += 1

        missing = [c for c in REQUIRED_COLUMNS if c not in row]
        if missing:
            stats["skipped_missing_columns"] += 1
            continue

        if str(row.get("type", "")).strip().lower() != "bounding box":
            stats["skipped_not_bbox_type"] += 1
            continue
        if str(row.get("category", "")).strip().lower() != "reference":
            # Deliberately excludes category=="point" (a different, incompatible task
            # shape -- see module docstring).
            stats["skipped_not_reference_category"] += 1
            continue

        # Real bug fix (2026-09-12): BigEarthNet.txt has its own official train/val/test
        # patch-pair splits (229,114/118,095/116,835 per arXiv:2603.29630), inherited from
        # BigEarthNet v2.0. Only draw training examples from the real "train" split --
        # otherwise this silently mixes in BigEarthNet.txt's own val/test rows, which is
        # sloppy regardless of whether it happens to contaminate any metric THIS project
        # currently reports (it doesn't -- our held-out eval is VRSBench-based, a separate
        # dataset), and would definitely contaminate any future BigEarthNet.txt-native eval
        # harness built from the real held-out split this project doesn't yet use.
        if str(row.get("split", "")).strip().lower() != "train":
            stats["skipped_not_train_split"] += 1
            continue

        patch_id = row.get("patch_id")
        if not patch_id:
            stats["skipped_no_patch_id"] += 1
            continue
        if per_patch_count[patch_id] >= dedupe_per_patch:
            # Cap examples-per-patch so a handful of heavily-annotated patches can't
            # dominate the subset and make the "diverse real imagery" claim dishonest.
            stats["skipped_per_patch_cap"] += 1
            continue

        prompt = clean_reference_prompt(row.get("input", ""))
        if prompt is None:
            stats["skipped_unparseable_prompt"] += 1
            continue

        box_token = box_str_to_token(row.get("output", ""))
        if box_token is None:
            stats["skipped_unparseable_or_invalid_box"] += 1
            continue

        selected.append({
            "patch_id": patch_id,
            # Real bug fix (2026-09-13): the live BigEarthNet.txt schema (confirmed
            # via this project's own 2026-09-13 Kaggle smoke-test run) has this column
            # as lowercase "s1_name", not "S1_name" -- the old capitalized lookup
            # silently returned None for every row. Not currently used downstream
            # (reserved for future SAR/fusion work), but fixed for honesty.
            "s1_name": row.get("s1_name"),
            "prompt": prompt,
            "response": box_token,
            "task_tag": "refer",
            "source": "bigearthnet_txt",
        })
        per_patch_count[patch_id] += 1
        stats["selected"] += 1

    return selected, stats


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--max-examples", type=int, default=2500,
                     help="Cap on selected referring-expression examples (default 2500 -- "
                          "see finetuning/README note on why this size was chosen).")
    ap.add_argument("--dedupe-per-patch", type=int, default=3)
    ap.add_argument("--out", type=Path, default=Path("bigearthnet_txt_selected.json"))
    ap.add_argument("--smoke-test", action="store_true",
                     help="Only scan the first 20,000 rows (fast sanity check before the "
                          "full 9.55M-row scan) -- mirrors this project's existing "
                          "SMOKE_TEST convention in kaggle_finetune_qwen2vl.ipynb.")
    args = ap.parse_args()

    # Real network access happens ONLY here, isolated to one place, so this whole module's
    # selection logic stays independently unit-testable without the `datasets` library.
    from datasets import load_dataset

    limit = 20_000 if args.smoke_test else None
    print(f"Loading {TEXT_DATASET_REPO} ({'smoke-test: first ' + str(limit) + ' rows' if limit else 'full 9.55M rows'})...")
    ds = load_dataset(TEXT_DATASET_REPO, split="all_data", streaming=True)
    rows = []
    for i, row in enumerate(ds):
        if limit and i >= limit:
            break
        rows.append(row)
        if (i + 1) % 500_000 == 0:
            print(f"  ...scanned {i + 1} rows")
    print(f"Scanned {len(rows)} real rows.")
    # Real bug fix (2026-09-13): release the streaming dataset/iterator handle as
    # soon as we're done consuming it. See the os._exit note at the end of main()
    # for why this alone isn't sufficient and what the real fix is.
    del ds

    if rows:
        print("Real column names found in first row:", list(rows[0].keys()))

    selected, stats = select_rows(rows, max_examples=args.max_examples,
                                   dedupe_per_patch=args.dedupe_per_patch)
    print(f"\nReal selection stats: {dict(stats)}")
    print(f"Selected {len(selected)} real referring-expression examples across "
          f"{len(set(r['patch_id'] for r in selected))} unique real patches.")

    assert len(selected) > 20, (
        "Too few real examples were selected -- this means either the real column names, "
        "the 'bounding box'/'reference' type/category values, or the box/ref text formats "
        "on the live data no longer match what was verified on 2026-08-29. Print "
        "rows[0] and a few type=='bounding box' rows raw, paste back, and fix this "
        "script's assumptions rather than lowering this assertion."
    )

    args.out.write_text(json.dumps(selected, indent=2))
    print(f"\nWrote {args.out} -- next: run finetuning/bigearthnet_txt_extract.py to fetch "
          f"the real images for these patch_ids and produce a training-ready manifest.json.")

    # Real bug fix (2026-09-13): everything real is now saved to disk. HF's `datasets`
    # streaming path (via fsspec/huggingface_hub's HTTP retry machinery) can leave a
    # background thread alive with an in-flight request/retry against the remote
    # parquet file, even though the foreground scan loop above already stopped
    # consuming it early (always true for --smoke-test, and also whenever
    # --max-examples caps selection before the full stream is exhausted). When the
    # interpreter then begins normal finalization, that leftover thread can call back
    # into the already-torn-down GIL state and crash with "Fatal Python error:
    # PyGILState_Release: auto-releasing thread-state, but no thread-state for this
    # thread" -- a real, observed crash (2026-09-13 Kaggle run) that happens AFTER the
    # real output above is already written correctly, i.e. it's a noisy false failure,
    # not a sign the real selection work was wrong. Fix: skip the normal interpreter
    # finalization that tries to join/clean up that stray thread, by exiting the
    # process immediately once all real work is safely on disk.
    sys.stdout.flush()
    sys.stderr.flush()
    os._exit(0)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile bigearthnet_txt_extract.py
"""
finetuning/bigearthnet_txt_extract.py

Takes bigearthnet_txt_selected.json (from bigearthnet_txt_filter.py -- a list of
{patch_id, s1_name, prompt, response, task_tag} rows with NO image yet) and produces a
real, training-ready manifest.json in the exact schema
finetuning/kaggle_finetune_qwen2vl.ipynb Cell 5 already expects
({"image_path", "prompt", "response", "task_tag"}) by fetching the REAL Sentinel-2 pixels
for each needed patch.

Why this is the hard part (verified via research on 2026-08-29, not assumed):
  - BigEarthNet v2.0's real images are NOT available as a per-patch/streamable download
    anywhere found (not on the official bigearth.net downloads page, not via a Planetary
    Computer STAC catalog -- an open unanswered request exists, not a live integration --
    and torchgeo's loader only handles the OLDER v1.0 archive with different patch IDs).
    The only real option is the official Zenodo release's whole-modality archives:
    BigEarthNet-S2.tar.zst (~59 GiB) and BigEarthNet-S1.tar.zst (~51 GiB).
  - Each patch is a REAL folder of 12 separate per-band GeoTIFFs for S2 (confirmed against
    bigearth.net's own "Description_BigEarthNet_v2.pdf"), not one combined image file --
    e.g. "S2B_MSIL2A_20180421T100029_N9999_R122_T33TWM_00_00_B04.tif" for the Red band.
  - There is no confirmed way to randomly seek into a .tar.zst for just the handful of
    patches we need -- zstd frames aren't seekable without a seekable-format archive, which
    this doesn't appear to be. So this script pays the FULL streaming-download cost once
    (~59 GiB of network transfer for S2-only, which is all this stage needs -- SAR/S1 is
    deliberately deferred, see module-level NOTE below) but keeps PEAK DISK USAGE small by
    reading the archive as a forward-only stream and only materializing the ~9 files per
    wanted patch (3 RGB bands here) it actually needs, discarding everything else without
    ever writing it to disk. Verified end-to-end against a synthetic .tar.zst fixture in
    test_bigearthnet_txt_extract.py (real network access to zenodo.org/bigearth.net is not
    reachable from the sandbox this was authored in -- confirmed via a direct connectivity
    check -- so this exact streaming logic could only be verified structurally there; it
    must be smoke-tested for real on Kaggle, which does have real network access, before
    trusting it against the full ~59 GiB run).

NOTE on scope (deliberate, not an oversight): this stage only extracts B04/B03/B02
(true-color RGB) from BigEarthNet-S2.tar.zst, NOT the S1 (SAR) archive. Reasons: (1) the
immediate ask is closing the Acc@0.5 REFERRING-EXPRESSION gap, which is an optical-only
grounding task; (2) Qwen2.5-VL's frozen CLIP-style vision tower expects standard RGB input,
not a 12-band multispectral tensor, so extracting more than 3 bands per S2 patch would be
wasted bandwidth for this stage regardless. Extending this same streaming pass to also
pull VV/VH from BigEarthNet-S1.tar.zst (for the separately-flagged real-SAR-fusion gap in
sar_fusion_service.py, which currently uses a PNG-proxy) is a natural follow-on, not
included here to keep this stage's ~59 GiB (not ~118 GiB) cost matched to what it actually
buys.

Cross-account/cross-session resume design: the expensive 59 GiB stream should only ever
happen ONCE, from whichever Kaggle account/session runs this first. After building the
small (images + manifest.json, expected tens of MB for ~2,500 small 120x120 patches) real
output, this script pushes it to a private HF Hub DATASET repo. Every subsequent run (any
Kaggle account, any session, as long as it has an HF_TOKEN with access to that same repo --
the same precondition the existing LoRA-checkpoint resume in Cell 6 already relies on)
checks that repo first and, if it already has files, just downloads the small cached
dataset instead of ever touching Zenodo again.
"""
from __future__ import annotations

import argparse
import io
import json
from pathlib import Path
from typing import Iterable

import numpy as np

S2_BANDS_RGB = ["B04", "B03", "B02"]  # Red, Green, Blue -- true-color order
S2_ARCHIVE_URL = "https://zenodo.org/records/10891137/files/BigEarthNet-S2.tar.zst"
CACHE_REPO_SUFFIX = "-bigearthnet-txt-cache"  # appended to the project's existing HF_REPO_ID


def wanted_member_names(patch_ids: Iterable[str]) -> dict[str, tuple[str, str]]:
    """Maps the exact basename we're looking for (e.g. "<patch_id>_B04.tif") to
    (patch_id, band) -- matched by BASENAME only, not full path, since this project does
    not have a verified exact root-folder prefix inside the real .tar.zst (only the
    per-patch-folder + per-band-file naming convention was verified via the official PDF).
    Matching by basename is robust to whatever the real root prefix turns out to be."""
    wanted = {}
    for pid in patch_ids:
        for band in S2_BANDS_RGB:
            wanted[f"{pid}_{band}.tif"] = (pid, band)
    return wanted


def stretch_band_to_uint8(band: np.ndarray, low_pct: float = 2.0, high_pct: float = 98.0) -> np.ndarray:
    """Real Sentinel-2 true-color rendering choice (disclosed, not arbitrary): raw S2
    reflectance bands are uint16 and mostly-dark by eye without a stretch. A 2-98
    percentile clip-and-rescale to 0-255 is the same convention widely used for S2
    true-color previews (e.g. Sentinel Hub / EO Browser's default rendering) -- chosen
    here for the same reason, not invented for this project."""
    band = band.astype(np.float32)
    lo, hi = np.percentile(band, [low_pct, high_pct])
    if hi <= lo:
        # Degenerate (e.g. a fully-uniform synthetic test band) -- avoid div-by-zero,
        # fall back to a flat mid-gray rather than crashing on real edge-case patches.
        return np.full_like(band, 128, dtype=np.uint8)
    stretched = np.clip((band - lo) / (hi - lo), 0.0, 1.0) * 255.0
    return stretched.astype(np.uint8)


def bands_to_rgb_image(band_arrays: dict[str, np.ndarray]):
    """band_arrays must have keys 'B04','B03','B02'. Returns a PIL.Image (RGB). Kept
    separate from any tar/network code so it's directly unit-testable with synthetic
    numpy arrays."""
    from PIL import Image
    missing = [b for b in S2_BANDS_RGB if b not in band_arrays]
    if missing:
        raise ValueError(f"missing required bands for RGB compose: {missing}")
    r = stretch_band_to_uint8(band_arrays["B04"])
    g = stretch_band_to_uint8(band_arrays["B03"])
    b = stretch_band_to_uint8(band_arrays["B02"])
    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray(rgb, mode="RGB")


def read_band_geotiff_bytes(raw_bytes: bytes) -> np.ndarray:
    """Reads one in-memory GeoTIFF band file's bytes into a 2D numpy array via rasterio's
    MemoryFile -- no temp file on disk needed for the per-band read itself."""
    import rasterio
    from rasterio.io import MemoryFile
    with MemoryFile(raw_bytes) as memfile:
        with memfile.open() as src:
            return src.read(1)


def _extract_from_zst_stream(raw_stream, wanted: dict[str, tuple[str, str]]) -> dict[str, dict[str, bytes]]:
    """Core forward-only stream-consuming logic, factored out from stream_extract_patches
    so it's directly testable against a local file object (no real network needed) -- see
    test_bigearthnet_txt_extract.py."""
    import tarfile
    import zstandard

    per_patch: dict[str, dict[str, bytes]] = {}
    wanted_patch_ids = {pid for pid, _band in wanted.values()}

    dctx = zstandard.ZstdDecompressor()
    members_seen = 0
    with dctx.stream_reader(raw_stream) as reader:
        with tarfile.open(fileobj=reader, mode="r|*") as tar:
            for member in tar:
                members_seen += 1
                basename = member.name.rsplit("/", 1)[-1]
                hit = wanted.get(basename)
                if not hit:
                    continue
                pid, band = hit
                f = tar.extractfile(member)
                if f is None:
                    continue
                per_patch.setdefault(pid, {})[band] = f.read()
                if members_seen % 200_000 == 0:
                    have = len(per_patch)
                    print(f"  ...scanned {members_seen} tar members, "
                          f"{have}/{len(wanted_patch_ids)} wanted patches matched so far")
                if len(per_patch) == len(wanted_patch_ids) and all(
                    len(v) == len(S2_BANDS_RGB) for v in per_patch.values()
                ):
                    print("All wanted patches fully matched -- stopping stream early "
                          "(no need to read the rest of the archive).")
                    break
    return per_patch


def stream_extract_patches(archive_url: str, wanted: dict[str, tuple[str, str]],
                            out_dir: Path, session=None) -> dict[str, dict[str, bytes]]:
    """Streams `archive_url` (a .tar.zst) forward-only and collects raw band-file bytes
    for members whose BASENAME is in `wanted`, grouped by patch_id. Never writes the bulk
    archive to disk -- only ever holds the (small) matched band bytes in memory before
    they're immediately composed into an RGB PNG and written out. Real HTTP streaming
    happens only when `session` is a real requests.Session."""
    if session is None:
        raise ValueError("a real requests.Session is required")
    resp = session.get(archive_url, stream=True, timeout=120)
    resp.raise_for_status()
    raw_stream = resp.raw
    raw_stream.decode_content = True
    return _extract_from_zst_stream(raw_stream, wanted)


def build_manifest(selected: list[dict], per_patch_bands: dict[str, dict[str, np.ndarray]],
                    images_dir: Path) -> list[dict]:
    """Pure logic (no I/O beyond writing the given images_dir) -- composes RGB images once
    per unique patch_id, then emits one manifest row per selected (prompt, response) pair,
    matching finetuning/kaggle_finetune_qwen2vl.ipynb Cell 3's manifest.json schema."""
    images_dir.mkdir(parents=True, exist_ok=True)
    image_path_by_patch: dict[str, str] = {}
    manifest = []
    skipped_no_image = 0
    for row in selected:
        pid = row["patch_id"]
        if pid not in image_path_by_patch:
            bands = per_patch_bands.get(pid)
            if not bands or any(b not in bands for b in S2_BANDS_RGB):
                skipped_no_image += 1
                continue
            img = bands_to_rgb_image(bands)
            img = img.resize((max(336, img.width), max(336, img.height)), resample=1)
            img_path = images_dir / f"ben_txt_{pid}.png"
            img.save(img_path)
            image_path_by_patch[pid] = str(img_path)
        manifest.append({
            "image_path": image_path_by_patch[pid],
            "prompt": row["prompt"],
            "response": row["response"],
            "task_tag": row.get("task_tag", "refer"),
        })
    if skipped_no_image:
        print(f"WARNING: {skipped_no_image} selected examples had no matching extracted "
              f"image (patch not found in the archive stream) and were dropped.")
    return manifest


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--selected", type=Path, default=Path("bigearthnet_txt_selected.json"))
    ap.add_argument("--out-dir", type=Path, default=Path("bigearthnet_txt_real"))
    ap.add_argument("--cache-repo", type=str, required=True,
                     help="HF dataset repo id to cache the small processed output in, e.g. "
                          "YOUR-HF-USERNAME/satquery-qwen25vl-vrsbench-lora-bigearthnet-txt-cache "
                          "(same HF account/token this project's LoRA checkpoints already use).")
    args = ap.parse_args()

    from huggingface_hub import HfApi, snapshot_download

    api = HfApi()
    try:
        api.dataset_info(args.cache_repo)
        print(f"Found an existing cached dataset at https://huggingface.co/datasets/{args.cache_repo} "
              f"-- downloading that directly instead of re-streaming the ~59 GiB Zenodo archive.")
        local = snapshot_download(repo_id=args.cache_repo, repo_type="dataset")
        cached_manifest = json.loads((Path(local) / "manifest.json").read_text())
        args.out_dir.mkdir(parents=True, exist_ok=True)
        (args.out_dir / "manifest.json").write_text(json.dumps(cached_manifest, indent=2))
        print(f"Wrote {args.out_dir}/manifest.json from cache ({len(cached_manifest)} examples). Done.")
        return
    except Exception as e:
        print(f"No cached dataset found at {args.cache_repo} ({e}) -- doing the real, "
              f"one-time ~59 GiB extraction from Zenodo. This will be cached for every "
              f"future run/account after this one.")

    import requests
    selected = json.loads(args.selected.read_text())
    patch_ids = sorted({row["patch_id"] for row in selected})
    wanted = wanted_member_names(patch_ids)
    print(f"Need {len(patch_ids)} unique real patches ({len(wanted)} band files) out of "
          f"{len(selected)} selected training examples.")

    session = requests.Session()
    raw_bands = stream_extract_patches(S2_ARCHIVE_URL, wanted, args.out_dir, session=session)
    per_patch_arrays = {
        pid: {band: read_band_geotiff_bytes(data) for band, data in bands.items()}
        for pid, bands in raw_bands.items()
    }

    images_dir = args.out_dir / "images"
    manifest = build_manifest(selected, per_patch_arrays, images_dir)
    assert len(manifest) > 20, (
        "Too few real images were matched -- likely means the tar member basename "
        "convention differs from what was assumed (<patch_id>_<band>.tif). Print the "
        "first ~20 real member names from the stream and compare before re-running "
        "against the full archive."
    )
    (args.out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    print(f"Wrote {args.out_dir}/manifest.json with {len(manifest)} real training examples.")

    print(f"Pushing small processed cache to https://huggingface.co/datasets/{args.cache_repo} "
          f"so no future run ever needs to re-stream the full archive...")
    api.create_repo(args.cache_repo, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(repo_id=args.cache_repo, repo_type="dataset", folder_path=str(args.out_dir))
    print("Done.")


if __name__ == "__main__":
    main()


## Smoke test first — fast sanity check (first 20,000 rows only, ~50 examples)

In [ ]:
!python bigearthnet_txt_filter.py --smoke-test --max-examples 50


**Check the printed output above before continuing**: it should show real column names, non-zero `selected` count in the stats dict, and no assertion error. The process now exits cleanly right after printing "Wrote bigearthnet_txt_selected.json" -- if you still see any error AFTER that line, paste it back before continuing (that would now be a genuinely new issue, not the known crash this version fixes).

## Real filter run — scans the full 9.55M-row BigEarthNet.txt annotation set

In [ ]:
!python bigearthnet_txt_filter.py --max-examples 150000


## HF login (needed for the extract step below, which pushes the cache to a private HF dataset repo)

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to Hugging Face Hub.")


## Real extract — streams the ~59 GiB Sentinel-2 archive ONCE, keeps only the small matched images, pushes the result to your private HF cache repo

This is the expensive step (time-wise). Peak disk usage stays small — only the handful of matched patch images are ever written to disk, never the bulk archive. If a later run (this account or another) finds the cache repo already populated, it skips straight to downloading the small cache instead of re-streaming.

In [ ]:
!python bigearthnet_txt_extract.py --cache-repo Sameelkazi/satquery-qwen3vl-vrsbench-lora-v4-bigearthnet-txt-cache


## Done

Once this finishes cleanly, the small `bigearthnet_txt_real/manifest.json` + images are cached at `https://huggingface.co/datasets/Sameelkazi/satquery-qwen3vl-vrsbench-lora-v4-bigearthnet-txt-cache`. When you run `kaggle_finetune_qwen2vl_v4.ipynb` (the GPU training notebook), Cell 6 will pick this up automatically as long as `bigearthnet_txt_real/manifest.json` exists in that session's working directory, or via that same cache-repo lookup — no extra step needed there.